Day 1 Complete Setup - PROF Baseline GRPO
==========================================
Complete implementation of:
- Unsloth GRPO setup
- Qwen2.5-Math-PRM-7B loading (4-bit)
- PROF Algorithm 1 filtering
- Smoke test

Copy-paste this entire file into Kaggle notebook and run cell-by-cell.
GPU: P100 (16GB), Internet: ON, Persistence: ON

In [1]:
# ============================================================================
# SECTION 1: DEPENDENCIES \u0026 ENVIRONMENT SETUP
# ============================================================================

print("=" * 60)
print("SECTION 1: Installing dependencies...")
print("=" * 60)

SECTION 1: Installing dependencies...


In [1]:
import sys
print(f"Python: {sys.version}")

# Step 1: Install core dependencies first (these are fast)
print("\n=== Step 1/4: Installing core packages ===")
!pip install -U datasets scipy transformers accelerate peft bitsandbytes

# Step 2: Install TRL with specific version
print("\n=== Step 2/4: Installing TRL ===")
!pip install "trl<0.9.0"

# Step 3: Install xformers (this can be slow, but shows progress)
print("\n=== Step 3/4: Installing xformers ===")
#!pip install "xformers<0.0.27"

# Step 4: Install Unsloth (from PyPI, faster than git)
print("\n=== Step 4/4: Installing Unsloth ===")
# Try PyPI first (faster)
try:
    !pip install unsloth
    print("✅ Unsloth installed from PyPI")
except:
    # Fallback to git if needed (with progress visible)
    print("⚠️ PyPI failed, installing from git (this may take 5-10 min)")
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("\n" + "="*60)
print("✅ ALL DEPENDENCIES INSTALLED")
print("="*60)

# Verify installations
import torch
import transformers
import unsloth
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, AutoConfig
import torch.nn.functional as F
from datasets import load_dataset
import re
import time
from datetime import datetime

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print(f"\nVersions:")
print(f"  PyTorch: {torch.__version__}")
print(f"  Transformers: {transformers.__version__}")
print(f"  Unsloth: {unsloth.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\n✅ Ready to proceed to Section 2")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

=== Step 1/4: Installing core packages ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 14.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 57.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 94.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 105.1 MB/s eta 0:00:0000:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalle

/tmp/ipykernel_57/791709373.py:34: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ PyTorch version: 2.10.0+cu128
✅ CUDA available: True
✅ GPU: Tesla T4
✅ VRAM: 15.64 GB

Versions:
  PyTorch: 2.10.0+cu128
  Transformers: 5.5.0
  Unsloth: 2026.5.2
  CUDA available: True
  GPU: Tesla T4
  VRAM: 15.64 GB

✅ Ready to proceed to Section 2


In [14]:
# ============================================================================
# SECTION 2: LOAD POLICY MODEL (Qwen2.5-Math-1.5B)
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 2: Loading policy model...")
print("=" * 60)

from unsloth import FastLanguageModel

# Load Qwen2.5-Math-1.5B in bf16 (should use ~6-7GB)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Math-1.5B",
    max_seq_length=4096,  # Per Ye et al. Section 4.1
    dtype=torch.bfloat16,  # P100 safe (no FP16 issues)
    load_in_4bit=False,    # Full precision training
)

print(f"✅ Model loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Add LoRA adapters (reduces memory, faster training)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,  # ← Change from "unsloth" to True
    random_state=3407,
)


# Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")


SECTION 2: Loading policy model...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Device does not support bfloat16. Will change to float16.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Model loaded. VRAM used: 13.57 GB
✅ Trainable: 18,464,768 / 1,562,179,072 (1.18%)


In [45]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [5]:
# ============================================================================
# SECTION 3: LOAD PRM MODEL (Qwen2.5-Math-PRM-7B in bf16)
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 3: Loading PRM model...")
print("=" * 60)

def load_prm_model(model_name="Qwen/Qwen2.5-Math-PRM-7B", device="auto"):
    """
    Load PRM in bf16 for inference (4-bit was causing NaN logits).
    Per Ye et al.: frozen PRM, no training, just scoring.

    Note: Using full bf16 instead of 4-bit quantization to avoid NaN outputs.
    This uses ~7GB VRAM instead of ~2-3GB, but is necessary for correct scoring.
    """
    print(f"  Loading PRM tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # CRITICAL FIX: The Qwen2RMConfig in config.json is missing pad_token_id
    # This causes AttributeError at line 814 of modeling_qwen2_rm.py
    # We MUST patch the config before model loading
    print(f"  Loading and patching config (pad_token_id missing from Qwen2RMConfig)...")
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

    # Set pad_token_id to eos_token_id (standard for Qwen models)
    # From config.json: eos_token_id = 151645
    config.pad_token_id = config.eos_token_id
    print(f"     Patched: config.pad_token_id = {config.pad_token_id}")

    # Load model with patched config
    model = AutoModel.from_pretrained(
        model_name,
        config=config,  # Pass patched config
        device_map=device,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    ).eval()

    # Freeze all parameters (no gradients needed)
    for param in model.parameters():
        param.requires_grad = False

    print(f"  PRM model loaded successfully")
    return model, tokenizer

# Load PRM
prm_model, prm_tokenizer = load_prm_model()
print(f"✅ PRM loaded. Total VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# DEBUG: Check model architecture and weights
print(f"\n[CRITICAL DEBUG] Model architecture check...")
print(f"  Model type: {type(prm_model).__name__}")
print(f"  Model has 'score' module: {hasattr(prm_model, 'score')}")
print(f"  Model has 'lm_head' module: {hasattr(prm_model, 'lm_head')}")
print(f"  Top-level modules: {[name for name, _ in prm_model.named_children()]}")

print(f"\n[CRITICAL DEBUG] Checking model weights for NaN...")
nan_params = []
total_params_checked = 0
for name, param in prm_model.named_parameters():
    total_params_checked += 1
    if torch.isnan(param).any():
        nan_params.append(name)
        print(f"  ❌ NaN found in: {name}")
    if total_params_checked <= 5:  # Show first 5 param stats
        print(f"  Param {name}: shape={param.shape}, mean={param.mean().item():.4f}, std={param.std().item():.4f}")

if nan_params:
    print(f"\n⚠️  CRITICAL: {len(nan_params)} parameters contain NaN!")
    print(f"   This explains the NaN logits - model weights are corrupted!")
else:
    print(f"  ✅ All {total_params_checked} model weights are valid (no NaN)")



SECTION 3: Loading PRM model...
  Loading PRM tokenizer...
  Loading and patching config (pad_token_id missing from Qwen2RMConfig)...
     Patched: config.pad_token_id = 151645


modeling_qwen2_rm.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Qwen/Qwen2.5-Math-PRM-7B:
- modeling_qwen2_rm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/342 [00:00<?, ?it/s]

Qwen2ForProcessRewardModel LOAD REPORT from: Qwen/Qwen2.5-Math-PRM-7B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  PRM model loaded successfully
✅ PRM loaded. Total VRAM used: 10.35 GB

[CRITICAL DEBUG] Model architecture check...
  Model type: Qwen2ForProcessRewardModel
  Model has 'score' module: True
  Model has 'lm_head' module: False
  Top-level modules: ['model', 'score']

[CRITICAL DEBUG] Checking model weights for NaN...
  Param model.embed_tokens.weight: shape=torch.Size([152064, 3584]), mean=-0.0000, std=0.0165
  Param model.layers.0.self_attn.q_proj.weight: shape=torch.Size([3584, 3584]), mean=-0.0000, std=0.0254
  Param model.layers.0.self_attn.q_proj.bias: shape=torch.Size([3584]), mean=0.1069, std=3.1094
  Param model.layers.0.self_attn.k_proj.weight: shape=torch.Size([512, 3584]), mean=-0.0000, std=0.0330
  Param model.layers.0.self_attn.k_proj.bias: shape=torch.Size([512]), mean=1.4219, std=28.0000
  ✅ All 342 model weights are valid (no NaN)


In [6]:
# ============================================================================
# SECTION 4: IMPLEMENT STEP-WISE PRM SCORING
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 4: Implementing PRM scoring functions...")
print("=" * 60)

def make_step_rewards(logits, token_masks):
    """
    Extract step-wise reward scores from PRM logits.

    From Qwen2.5-Math-PRM-7B model card implementation.

    Args:
        logits: Model output logits of shape (batch_size, seq_length, 2)
        token_masks: Boolean mask marking step separator positions

    Returns:
        List of lists where each inner list contains step rewards [0-1]
    """
    probabilities = F.softmax(logits, dim=-1)
    probabilities = probabilities * token_masks.unsqueeze(-1)

    all_scores_res = []
    for i in range(probabilities.size(0)):
        sample = probabilities[i]
        # Extract positive class probability at step separators
        positive_probs = sample[sample != 0].view(-1, 2)[:, 1]
        non_zero_elements_list = positive_probs.cpu().tolist()
        all_scores_res.append(non_zero_elements_list)
    return all_scores_res


def compute_step_rewards(response_text, model, tokenizer):
    """
    Compute step-wise PRM scores for a response.

    Per Ye et al. Section 3.1: Steps delimited by newlines.

    Args:
        response_text: Full response text from policy model
        model: Loaded PRM model
        tokenizer: Corresponding tokenizer

    Returns:
        step_rewards: List of reward scores for each step [0-1]
    """
    # Split response into steps by double newline (Ye et al. approach)
    steps = response_text.split('\n\n')

    # Filter out empty steps
    steps = [s.strip() for s in steps if s.strip()]

    if not steps:
        return []

    # Format for chat template with <extra_0> step markers
    # This is how Qwen2.5-Math-PRM expects input
    messages = [
        {"role": "system", "content": "Please reason step by step."},
        {"role": "user", "content": "Solve this problem."},
        {"role": "assistant", "content": "<extra_0>".join(steps) + "<extra_0>"},
    ]

    # Apply chat template and tokenize
    conversation_str = prm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Tokenize with attention_mask (CRITICAL for avoiding NaN)
    tokenized = prm_tokenizer(
        conversation_str,
        return_tensors="pt",
        padding=False,  # No padding needed for single sequence
    )
    input_ids = tokenized['input_ids'].to(prm_model.device)
    attention_mask = tokenized['attention_mask'].to(prm_model.device)

    # Forward pass through PRM (no gradients)
    # CRITICAL: Must pass attention_mask to avoid NaN in attention computation
    with torch.no_grad():
        outputs = prm_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False
        )

    # Debug: Check output structure
    print(f"  [DEBUG] Output type: {type(outputs)}")
    print(f"  [DEBUG] Output keys: {outputs.keys() if hasattr(outputs, 'keys') else 'N/A'}")

    # Try to get logits - different models have different output structures
    if hasattr(outputs, 'logits'):
        logits = outputs.logits
        print(f"  [DEBUG] Using outputs.logits")
    elif isinstance(outputs, tuple) and len(outputs) > 0:
        logits = outputs[0]
        print(f"  [DEBUG] Using outputs[0]")
    else:
        logits = outputs
        print(f"  [DEBUG] Using outputs directly")

    print(f"  [DEBUG] Logits shape: {logits.shape}")
    print(f"  [DEBUG] Logits dtype: {logits.dtype}")
    print(f"  [DEBUG] Logits device: {logits.device}")
    print(f"  [DEBUG] Logits contains NaN: {torch.isnan(logits).any()}")
    print(f"  [DEBUG] Logits contains Inf: {torch.isinf(logits).any()}")
    print(f"  [DEBUG] Logits min/max: {logits.min().item():.4f} / {logits.max().item():.4f}")

    # Extract rewards at step separator positions
    step_sep_id = prm_tokenizer.encode("<extra_0>")[0]
    token_masks = (input_ids == step_sep_id)

    print(f"  [DEBUG] Step separator ID: {step_sep_id}")
    print(f"  [DEBUG] Number of step separators found: {token_masks.sum().item()}")

    step_rewards = make_step_rewards(logits, token_masks)

    return step_rewards[0] if step_rewards else []

# Test PRM scoring - using format closer to official example
print("\n[TEST] Testing PRM scoring with official example format...")
test_data = {
    "system": "Please reason step by step.",
    "query": "What is 2+2?",
    "response": [
        "Let me solve this step by step.",
        "Step 1: Add the two numbers together.",
        "Step 2: 2 + 2 = 4.",
        "The answer is 4."
    ]
}

test_messages = [
    {"role": "system", "content": test_data['system']},
    {"role": "user", "content": test_data['query']},
    {"role": "assistant", "content": "<extra_0>".join(test_data['response']) + "<extra_0>"},
]

test_conversation = prm_tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False
)

print(f"  Conversation length: {len(test_conversation)} chars")
test_tokenized = prm_tokenizer(test_conversation, return_tensors="pt", padding=False)
test_input_ids = test_tokenized['input_ids'].to(prm_model.device)
test_attention_mask = test_tokenized['attention_mask'].to(prm_model.device)
print(f"  Input IDs shape: {test_input_ids.shape}")

with torch.no_grad():
    test_outputs = prm_model(
        input_ids=test_input_ids,
        attention_mask=test_attention_mask,
        use_cache=False
    )

test_step_sep_id = prm_tokenizer.encode("<extra_0>")[0]
test_token_masks = (test_input_ids == test_step_sep_id)
print(f"  Step separators found: {test_token_masks.sum().item()}")

test_logits = test_outputs.logits if hasattr(test_outputs, 'logits') else test_outputs[0]
print(f"  Logits shape: {test_logits.shape}")
print(f"  Logits contains NaN: {torch.isnan(test_logits).any().item()}")

if not torch.isnan(test_logits).any():
    test_step_rewards = make_step_rewards(test_logits, test_token_masks)
    print(f"✅ Test PRM scores: {test_step_rewards}")
    print(f"   Expected: ~4 scores, each in [0, 1]")
else:
    print(f"❌ Test FAILED: Logits contain NaN")
    print(f"   Logits min/max: {test_logits.min().item():.4f} / {test_logits.max().item():.4f}")


`use_return_dict` is deprecated! Use `return_dict` instead!



SECTION 4: Implementing PRM scoring functions...

[TEST] Testing PRM scoring with official example format...
  Conversation length: 280 chars
  Input IDs shape: torch.Size([1, 69])


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  Step separators found: 4
  Logits shape: torch.Size([1, 69, 2])
  Logits contains NaN: False
✅ Test PRM scores: [[0.197265625, 0.2265625, 0.2431640625, 0.2578125]]
   Expected: ~4 scores, each in [0, 1]


In [7]:
# ============================================================================
# SECTION 5: IMPLEMENT PROF ALGORITHM 1 (FILTERING)
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 5: Implementing PROF Algorithm 1...")
print("=" * 60)

def compute_trajectory_prm_score(step_rewards, outcome_reward,
                                lambda_reg=10, H_lambda=30):
    """
    Compute trajectory-level PRM score with step regularization.

    Implements PROF Algorithm 1, Equation 1:
    r_pro_i = [mean(step_rewards) - λ·I(H=1 or H≥H_λ)] · ro,i

    Args:
        step_rewards: List of step-wise PRM scores (each in [0, 1])
        outcome_reward: Outcome reward {-1, 1} (incorrect/correct)
        lambda_reg: Regularization weight λ=10 (Ye et al.)
        H_lambda: Threshold H_λ=30 (Ye et al.)

    Returns:
        trajectory_score: Float representing consistency
    """
    num_steps = len(step_rewards)

    # Trajectory-level score = mean of step-wise rewards
    if num_steps == 0:
        mean_step_reward = 0.0
    else:
        mean_step_reward = sum(step_rewards) / num_steps

    # Step length regularization: penalize trivial (1 step) or very long (>H_λ)
    step_penalty = 0.0
    if num_steps == 1 or num_steps >= H_lambda:
        step_penalty = lambda_reg

    # Final consistency score weighted by outcome reward
    trajectory_score = (mean_step_reward - step_penalty) * outcome_reward

    return trajectory_score


def prof_filter(rollouts, outcome_rewards, prm_model, prm_tokenizer,
                policy_update_size=4, lambda_reg=10, H_lambda=30):
    """
    Implement PROF filtering Algorithm 1 (complete).

    Filters n=8 rollouts down to m=4 based on consistency between
    PRM (process rewards) and ORM (outcome rewards).

    Args:
        rollouts: List of n=8 response texts
        outcome_rewards: List of n=8 outcome rewards {-1, 1}
        prm_model: Loaded PRM for step scoring
        prm_tokenizer: PRM tokenizer
        policy_update_size: Target m=4 (Ye et al.)

    Returns:
        filtered_rollouts: List of m=4 kept responses
        filtered_rewards: List of m=4 corresponding rewards
        filter_stats: Dict with debugging info
    """
    n = len(rollouts)
    n_correct = sum(1 for r in outcome_rewards if r == 1)
    n_incorrect = n - n_correct
    delta = n_correct - n_incorrect

    # Step 1: Compute PRM step scores for each rollout
    all_step_rewards = []
    for response in rollouts:
        step_rewards = compute_step_rewards(response, prm_model, prm_tokenizer)
        all_step_rewards.append(step_rewards)

    # Step 2: Compute trajectory-level consistency scores (Eq. 1)
    trajectory_scores = []
    for step_rewards, outcome in zip(all_step_rewards, outcome_rewards):
        score = compute_trajectory_prm_score(
            step_rewards, outcome, lambda_reg, H_lambda
        )
        trajectory_scores.append(score)

    # Step 3: Separate into correct (G+) and incorrect (G-) groups
    correct_indices = [i for i, r in enumerate(outcome_rewards) if r == 1]
    incorrect_indices = [i for i, r in enumerate(outcome_rewards) if r == -1]

    correct_scores = [(trajectory_scores[i], i) for i in correct_indices]
    incorrect_scores = [(trajectory_scores[i], i) for i in incorrect_indices]

    # Step 4: Calculate removal counts (Eq. 2)
    k_plus = min(n - policy_update_size,
                 int((delta + n - policy_update_size) / 2))
    k_minus = n - policy_update_size - k_plus

    # Step 5: Rank and filter
    # G+ (correct): Keep highest PRM scores (best reasoning among correct)
    # G- (incorrect): Keep lowest PRM scores (most obviously wrong)
    correct_ranked = sorted(correct_scores, key=lambda x: x[0], reverse=True)
    incorrect_ranked = sorted(incorrect_scores, key=lambda x: x[0], reverse=False)

    # Keep top from each group
    kept_correct_indices = [idx for _, idx in correct_ranked[:(n_correct - k_plus)]]
    kept_incorrect_indices = [idx for _, idx in incorrect_ranked[:(n_incorrect - k_minus)]]

    filtered_indices = kept_correct_indices + kept_incorrect_indices

    # Step 6: Build filtered outputs
    filtered_rollouts = [rollouts[i] for i in filtered_indices]
    filtered_rewards = [outcome_rewards[i] for i in filtered_indices]

    filter_stats = {
        'original_count': n,
        'kept_count': len(filtered_indices),
        'correct_removed': k_plus,
        'incorrect_removed': k_minus,
        'correct_kept': len(kept_correct_indices),
        'incorrect_kept': len(kept_incorrect_indices),
        'avg_prm_score_kept': sum(trajectory_scores[i] for i in filtered_indices) / len(filtered_indices) if filtered_indices else 0,
    }

    return filtered_rollouts, filtered_rewards, filter_stats

# Test filtering on dummy data
test_rollouts = ["Response A", "Response B", "Response C", "Response D",
                 "Response E", "Response F", "Response G", "Response H"]
test_outcomes = [1, 1, 1, 1, -1, -1, -1, -1]  # 4 correct, 4 incorrect

filtered, filtered_rewards, stats = prof_filter(
    test_rollouts, test_outcomes, prm_model, prm_tokenizer
)

print(f"\n✅ PROF Filtering Test:")
print(f"   Original: {len(test_rollouts)} rollouts")
print(f"   Filtered: {len(filtered)} rollouts (should be 4)")
print(f"   Stats: {stats}")

assert len(filtered) == 4, "PROF should keep m=4 rollouts!"
print("✅ PROF filtering validation passed!")



SECTION 5: Implementing PROF Algorithm 1...
  [DEBUG] Output type: <class 'transformers.modeling_outputs.TokenClassifierOutput'>
  [DEBUG] Output keys: odict_keys(['logits'])
  [DEBUG] Using outputs.logits
  [DEBUG] Logits shape: torch.Size([1, 29, 2])
  [DEBUG] Logits dtype: torch.bfloat16
  [DEBUG] Logits device: cuda:0
  [DEBUG] Logits contains NaN: False
  [DEBUG] Logits contains Inf: False
  [DEBUG] Logits min/max: -1.3047 / 1.0859
  [DEBUG] Step separator ID: 151651
  [DEBUG] Number of step separators found: 1
  [DEBUG] Output type: <class 'transformers.modeling_outputs.TokenClassifierOutput'>
  [DEBUG] Output keys: odict_keys(['logits'])
  [DEBUG] Using outputs.logits
  [DEBUG] Logits shape: torch.Size([1, 29, 2])
  [DEBUG] Logits dtype: torch.bfloat16
  [DEBUG] Logits device: cuda:0
  [DEBUG] Logits contains NaN: False
  [DEBUG] Logits contains Inf: False
  [DEBUG] Logits min/max: -1.3047 / 1.0859
  [DEBUG] Step separator ID: 151651
  [DEBUG] Number of step separators found: 1

In [8]:
# ============================================================================
# SECTION 6: LOAD TRAINING DATA
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 6: Loading training data...")
print("=" * 60)

# NuminaMath dataset (per Ye et al. paper)
# If not available, use GSM8K as proxy
try:
    train_dataset = load_dataset("AI-MO/NuminaMath-CoT", split="train")
    print("✅ Loaded NuminaMath-CoT dataset")
except:
    print("⚠️  NuminaMath not found, using GSM8K as proxy")
    train_dataset = load_dataset("openai/gsm8k", "main", split="train")

# Format for math reasoning
def format_prompt(example):
    """Convert dataset example to Qwen2.5-Math format"""
    # Format: "Question: {problem}\nAnswer:"
    if "problem" in example:
        problem = example["problem"]
    elif "question" in example:
        problem = example["question"]
    else:
        raise KeyError("No problem/question field in dataset")

    return f"Question: {problem}\nAnswer:"

# Prepare prompts (1024 per Ye et al.)
train_prompts = [format_prompt(ex) for ex in train_dataset.select(range(min(1024, len(train_dataset))))]
print(f"✅ Loaded {len(train_prompts)} training prompts")
print(f"   Example prompt:\n   {train_prompts[0][:150]}...")



SECTION 6: Loading training data...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

...

data/train-00004-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/166k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/859494 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

✅ Loaded NuminaMath-CoT dataset
✅ Loaded 1024 training prompts
   Example prompt:
   Question: Consider the terms of an arithmetic sequence: $-\frac{1}{3}, y+2, 4y, \ldots$. Solve for $y$.
Answer:...


In [9]:
# ============================================================================
# SECTION 7: IMPLEMENT BINARY OUTCOME REWARD (VERIFIER)
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 7: Implementing binary outcome reward...")
print("=" * 60)

def extract_answer(text):
    """Extract final numerical answer from model output"""
    # Look for patterns like "The answer is X" or "####X" (GSM8K format)
    patterns = [
        r'####\s*([0-9,\.]+)',           # GSM8K format
        r'[Tt]he answer is[:\s]*([0-9,\.]+)',
        r'[Aa]nswer[:\s]*([0-9,\.]+)',
        r'\\boxed\{([^}]+)\}',            # LaTeX boxed
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1).replace(',', '').strip()

    # Fallback: last number in text
    numbers = re.findall(r'[0-9,\.]+', text)
    return numbers[-1].replace(',', '').strip() if numbers else None

def binary_outcome_reward(prompt, response, ground_truth):
    """
    Binary outcome reward: +1 if correct, -1 if incorrect
    Per Ye et al.: ro ∈ {-1, +1}
    """
    predicted = extract_answer(response)

    if predicted is None:
        return -1.0  # No answer found

    # Normalize both answers
    try:
        pred_num = float(predicted)
        gt_num = float(ground_truth)
        is_correct = abs(pred_num - gt_num) < 1e-4
    except ValueError:
        # String comparison fallback
        is_correct = predicted.strip().lower() == str(ground_truth).strip().lower()

    return 1.0 if is_correct else -1.0

# Test on example
test_response = "Let me solve this step by step.\n1. First...\n2. Then...\nThe answer is 42."
test_reward = binary_outcome_reward("", test_response, "42")
print(f"✅ Test reward: {test_reward} (should be 1.0)")


SECTION 7: Implementing binary outcome reward...
✅ Test reward: 1.0 (should be 1.0)


In [15]:
# ============================================================================
# SECTION 8: CONFIGURE GRPO TRAINER (BASELINE)
# ============================================================================
print("\n" + "=" * 60)
print("SECTION 8: Configuring GRPO trainer...")
print("=" * 60)
from trl import GRPOConfig, GRPOTrainer

# Hyperparameters from Ye et al. Section 4.1
grpo_config = GRPOConfig(
    output_dir="./baseline_grpo_checkpoints",
    num_train_epochs=1,
    max_steps=1,                          # ← move here for smoke test
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-6,
    optim="adamw_torch",
    warmup_steps=10,
    num_generations=8,
    temperature=1.0,
    beta=0.001,
    epsilon=0.2,
    epsilon_high=0.28,
    max_completion_length=4096,
    max_prompt_length=512,
    logging_steps=10,
    save_steps=200,
    save_total_limit=5,
    fp16=True,
    gradient_checkpointing=False,  # Disable trainer-level checkpointing too
    report_to="none",
)
print("✅ GRPO config loaded:")
print(f"   Effective batch size: {grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps}")
print(f"   Generations per prompt: {grpo_config.num_generations}")
print(f"   Learning rate: {grpo_config.learning_rate}")


SECTION 8: Configuring GRPO trainer...
✅ GRPO config loaded:
   Effective batch size: 16
   Generations per prompt: 8
   Learning rate: 1e-06


In [16]:
# ============================================================================
# SECTION 9: SMOKE TEST - Single Forward + Backward Pass
# ============================================================================

print("\n" + "=" * 60)
print("SECTION 9: RUNNING SMOKE TEST...")
print("=" * 60)
print("⚠️  CRITICAL: This must pass before proceeding to Day 2")
print("=" * 60)

# Create minimal trainer for smoke test
trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    processing_class=tokenizer,
    reward_funcs=[
        lambda prompts, completions, **kwargs: [
            binary_outcome_reward(p, r, "42") for p, r in zip(prompts, completions)
        ]
    ],                                        # ← list, not bare lambda; accept **kwargs
    train_dataset=[{"prompt": p} for p in train_prompts[:10]],
)

# Clear VRAM before test
torch.cuda.empty_cache()
initial_memory = torch.cuda.memory_allocated() / 1e9

# Run 1 training step
print("\n🔥 Starting smoke test (1 training step)...")
print("   This will:")
print("   1. Generate n=8 rollouts per prompt")
print("   2. Compute rewards")
print("   3. Run GRPO loss backward pass")
print("   4. Update weights")
print()

try:
    trainer.train()

    peak_memory = torch.cuda.max_memory_allocated() / 1e9
    print(f"\n{'='*60}")
    print(f"✅ SMOKE TEST PASSED")
    print(f"{'='*60}")
    print(f"   Initial VRAM: {initial_memory:.2f} GB")
    print(f"   Peak VRAM: {peak_memory:.2f} GB")
    print(f"   Available headroom: {16 - peak_memory:.2f} GB")
    print(f"{'='*60}")

    if peak_memory > 15.0:
        print(f"⚠️  WARNING: Very tight memory ({peak_memory:.2f}/16 GB)")
        print(f"   Consider: reduce batch_size or num_generations")
    else:
        print(f"✅ Memory footprint looks good!")

    print(f"\n{'='*60}")
    print(f"🎉 DAY 1 COMPLETE - ALL GATES PASSED")
    print(f"{'='*60}")
    print(f"✅ Policy model loaded (~7GB)")
    print(f"✅ PRM model loaded (~3GB)")
    print(f"✅ PRM scoring works")
    print(f"✅ PROF filtering works (8 → 4)")
    print(f"✅ GRPO training loop works")
    print(f"✅ Peak VRAM: {peak_memory:.2f} GB (safe margin)")
    print(f"\n🚀 READY FOR DAY 2: Launch full training!")
    print(f"{'='*60}")

except RuntimeError as e:
    if "out of memory" in str(e):
        print(f"\n{'='*60}")
        print(f"❌ SMOKE TEST FAILED: OOM")
        print(f"{'='*60}")
        print(f"   Peak VRAM before crash: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
        print(f"\n🔧 Troubleshooting:")
        print(f"   1. Reduce num_generations: 8 → 4")
        print(f"   2. Reduce gradient_accumulation_steps: 16 → 8")
        print(f"   3. Reduce max_new_tokens: 4096 → 2048")
        print(f"   4. Enable load_in_4bit=True (last resort)")
        print(f"{'='*60}")
        raise
    else:
        print(f"\n{'='*60}")
        print(f"❌ SMOKE TEST FAILED: {e}")
        print(f"{'='*60}")
        raise


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



SECTION 9: RUNNING SMOKE TEST...
⚠️  CRITICAL: This must pass before proceeding to Day 2

🔥 Starting smoke test (1 training step)...
   This will:
   1. Generate n=8 rollouts per prompt
   2. Compute rewards
   3. Run GRPO loss backward pass
   4. Update weights



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
Both `max_new_tokens` (=4096) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/pytho

Step,Training Loss


Unsloth: Restored added_tokens_decoder metadata in ./baseline_grpo_checkpoints/checkpoint-1/tokenizer_config.json.



✅ SMOKE TEST PASSED
   Initial VRAM: 13.64 GB
   Peak VRAM: 14.90 GB
   Available headroom: 1.10 GB
✅ Memory footprint looks good!

🎉 DAY 1 COMPLETE - ALL GATES PASSED
✅ Policy model loaded (~7GB)
✅ PRM model loaded (~3GB)
✅ PRM scoring works
✅ PROF filtering works (8 → 4)
✅ GRPO training loop works
✅ Peak VRAM: 14.90 GB (safe margin)

🚀 READY FOR DAY 2: Launch full training!


In [17]:
import torch
import time
from datetime import datetime
from datasets import load_dataset
import random
import json

print("=" * 60)
print("DAY 2: BASELINE GRPO FULL TRAINING")
print("=" * 60)
print(f"Started: {datetime.now()}")
print("=" * 60)

DAY 2: BASELINE GRPO FULL TRAINING
Started: 2026-05-15 03:43:48.480517


In [33]:
import re

# ============================================================================
# NUMINAMATH-SPECIFIC ANSWER EXTRACTION (overrides Day 1 functions)
# ============================================================================

def extract_boxed(text):
    """
    Extract content from \boxed{...}, handling nested braces.
    NuminaMath solutions end with \boxed{answer}.
    """
    # Find \boxed{ and extract content with balanced braces
    pattern = r'\\boxed\{'
    match = re.search(pattern, text)
    if not match:
        return None

    start = match.end()
    depth = 1
    i = start
    while i < len(text) and depth > 0:
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
        i += 1

    if depth == 0:
        return text[start:i-1].strip()
    return None


def normalize_latex(expr):
    """
    Normalize LaTeX expression for comparison.
    Removes whitespace, normalizes common variations.
    """
    if expr is None:
        return None

    # Remove all whitespace
    expr = re.sub(r'\s+', '', expr)

    # Normalize common LaTeX variations
    expr = expr.replace('\\cdot', '*')
    expr = expr.replace('\\times', '*')
    expr = expr.replace('\\div', '/')
    expr = expr.replace('\\frac', 'frac')
    expr = expr.replace('\\dfrac', 'frac')
    expr = expr.replace('\\tfrac', 'frac')

    return expr.lower()


def extract_answer(text):
    """
    Extract answer from model output - NuminaMath compatible.
    Priority: \boxed{} > #### > "answer is" > last number
    """
    if text is None:
        return None

    # 1. Try \boxed{} first (NuminaMath format)
    boxed = extract_boxed(text)
    if boxed:
        return boxed

    # 2. GSM8K format: #### answer
    gsm_match = re.search(r'####\s*(.+?)(?:\n|$)', text)
    if gsm_match:
        return gsm_match.group(1).strip()

    # 3. "The answer is X" - capture full expression
    ans_match = re.search(r'[Tt]he (?:final )?answer is[:\s]*(.+?)(?:\.|$)', text)
    if ans_match:
        return ans_match.group(1).strip()

    # 4. Fallback: last \boxed-like pattern or number
    # Look for any remaining mathematical expression
    return None


def extract_ground_truth_numina(example):
    """
    Extract ground truth from NuminaMath-CoT dataset.
    The 'solution' field contains step-by-step with \boxed{answer} at end.
    """
    if "solution" in example and example["solution"]:
        boxed = extract_boxed(str(example["solution"]))
        if boxed:
            return boxed

    # Fallback to 'answer' field if exists
    if "answer" in example and example["answer"]:
        return str(example["answer"]).strip()

    return None


def binary_outcome_reward(prompt, response, ground_truth):
    """
    Binary outcome reward for NuminaMath: +1 if correct, -1 if incorrect.
    Compares normalized LaTeX expressions.
    """
    if ground_truth is None:
        return -1.0

    # Extract answer from model response
    predicted = extract_answer(response)

    if predicted is None:
        return -1.0

    # Normalize both for comparison
    pred_norm = normalize_latex(predicted)
    gt_norm = normalize_latex(ground_truth)

    if pred_norm is None or gt_norm is None:
        return -1.0

    # String comparison (exact match after normalization)
    if pred_norm == gt_norm:
        return 1.0

    # Try numeric comparison as fallback
    try:
        pred_num = float(re.sub(r'[^\d.\-]', '', predicted))
        gt_num = float(re.sub(r'[^\d.\-]', '', ground_truth))
        if abs(pred_num - gt_num) < 1e-6:
            return 1.0
    except (ValueError, TypeError):
        pass

    return -1.0

In [19]:
# ============================================================================
# SECTION 1: LOAD FULL TRAINING DATASET
# ============================================================================

print("\n" + "=" * 60)
print("Loading full training dataset...")
print("=" * 60)

# Load full NuminaMath/GSM8K (not just 10 samples)
try:
    train_dataset_full = load_dataset("AI-MO/NuminaMath-CoT", split="train")
    print(f"✅ Loaded NuminaMath-CoT: {len(train_dataset_full)} examples")
except:
    print("⚠️  NuminaMath not available, using GSM8K")
    train_dataset_full = load_dataset("openai/gsm8k", "main", split="train")
    print(f"✅ Loaded GSM8K: {len(train_dataset_full)} examples")

# Per Ye et al.: 1024 prompts per iteration
# Sample randomly if dataset > 1024
random.seed(42)
if len(train_dataset_full) > 1024:
    train_indices = random.sample(range(len(train_dataset_full)), 1024)
    train_dataset_full = train_dataset_full.select(train_indices)

train_prompts_full = [format_prompt(ex) for ex in train_dataset_full]
print(f"✅ Full training set: {len(train_prompts_full)} prompts")


Loading full training dataset...
✅ Loaded NuminaMath-CoT: 859494 examples
✅ Full training set: 1024 prompts


In [34]:
# ============================================================================
# SECTION 2: EXTRACT GROUND TRUTH ANSWERS
# ============================================================================

print("\n" + "=" * 60)
print("Extracting ground truth answers...")
print("=" * 60)

ground_truths = [extract_ground_truth_numina(ex) for ex in train_dataset_full]

# Check extraction quality
none_count = sum(1 for gt in ground_truths if gt is None)
print(f"   Extracted: {len(ground_truths) - none_count} valid, {none_count} failed")
print(f"✅ Extracted {len(ground_truths)} ground truth answers")
print(f"   Sample GT: {ground_truths[:3]}")

# Create lookup dict: prompt → ground_truth
prompt_to_gt = dict(zip(train_prompts_full, ground_truths))


Extracting ground truth answers...
   Extracted: 984 valid, 40 failed
✅ Extracted 1024 ground truth answers
   Sample GT: ['3^{n-1}', '-\\frac{2}{7}', '(-\\infty, -3) \\cup (5, +\\infty)']


In [35]:
# ============================================================================
# SECTION 3: CREATE REWARD FUNCTION WITH GROUND TRUTH LOOKUP
# ============================================================================

print("\n" + "=" * 60)
print("Creating reward function...")
print("=" * 60)

def reward_function_with_gt(prompts, completions, **kwargs):
    """Wrapper reward function for GRPO trainer"""
    rewards = []
    for prompt, completion in zip(prompts, completions):
        gt = prompt_to_gt.get(prompt)
        if gt is None:
            print(f"⚠️  Warning: No GT for prompt: {prompt[:50]}...")
            rewards.append(-1.0)
        else:
            reward = binary_outcome_reward(prompt, completion, gt)
            rewards.append(reward)
    return rewards

# Test on first few
test_rewards = reward_function_with_gt(train_prompts_full[:3], ["The answer is X"]*3)
print(f"✅ Test rewards: {test_rewards}")


Creating reward function...
✅ Test rewards: [-1.0, -1.0, -1.0]


In [36]:
# Test extraction and reward
print("\n[DEBUG] Testing NuminaMath extraction:")
test_prompt = train_prompts_full[0]
test_gt = ground_truths[0]
print(f"   Ground truth: '{test_gt}'")

# Test with correct boxed answer
test_response_correct = f"Step 1: solve\\nStep 2: compute\\nThe answer is \\boxed{{{test_gt}}}"
test_response_wrong = "The answer is \\boxed{999}"

reward_correct = binary_outcome_reward(test_prompt, test_response_correct, test_gt)
reward_wrong = binary_outcome_reward(test_prompt, test_response_wrong, test_gt)

print(f"   Correct response reward: {reward_correct} (should be 1.0)")
print(f"   Wrong response reward: {reward_wrong} (should be -1.0)")

if reward_correct == 1.0 and reward_wrong == -1.0:
    print("✅ Reward function working correctly!")
else:
    print("❌ Reward function needs debugging")


[DEBUG] Testing NuminaMath extraction:
   Ground truth: '3^{n-1}'
   Correct response reward: 1.0 (should be 1.0)
   Wrong response reward: -1.0 (should be -1.0)
✅ Reward function working correctly!


In [25]:
# ============================================================================
# SECTION 4: CREATE FULL BASELINE GRPO TRAINER
# ============================================================================

print("\n" + "=" * 60)
print("Creating full Baseline GRPO trainer...")
print("=" * 60)

from trl import GRPOTrainer

# Use same config as smoke test, but full dataset
trainer_baseline = GRPOTrainer(
    model=model,
    args=grpo_config,  # Same config from Day 1
    processing_class=tokenizer,
    reward_funcs=[reward_function_with_gt],  # Must be a list
    train_dataset=[{"prompt": p} for p in train_prompts_full],  # List of dicts
)

# Save initial model (for recovery)
model.save_pretrained("./baseline_grpo_checkpoints/step_0")
tokenizer.save_pretrained("./baseline_grpo_checkpoints/step_0")

print(f"✅ Trainer created")
print(f"   Training on: {len(train_prompts_full)} prompts")
print(f"   Generations per prompt: {grpo_config.num_generations}")


Creating full Baseline GRPO trainer...
✅ Trainer created
   Training on: 1024 prompts
   Generations per prompt: 8


In [31]:
# ============================================================================
# SECTION 5: START TRAINING WITH MONITORING
# ============================================================================

print("\n" + "=" * 60)
print("🚀 BASELINE GRPO TRAINING STARTED")
print("=" * 60)
print(f"   Time: {datetime.now()}")
print(f"   Dataset: {len(train_prompts_full)} prompts")
print(f"   Generations per prompt: {grpo_config.num_generations}")
print(f"   Expected duration: ~30-40 GPU hours")
print(f"   Checkpoints: every 200 steps → ./baseline_grpo_checkpoints/")
print("=" * 60)
print()
print("⚠️  MONITORING CHECKLIST (check every ~2 hours):")
print("   ✅ Training loss decreasing")
print("   ✅ VRAM stable (not creeping up)")
print("   ✅ Reward distribution: mix of +1.0 and -1.0")
print("   ✅ Checkpoints saving every 200 steps")
print("   ⚠️  If loss is NaN → stop and debug")
print("   ⚠️  If all rewards are -1 → verifier broken")
print("=" * 60)

start_time = time.time()

try:
    # This will run for many hours - monitor via Kaggle notebook logs
    trainer_baseline.train()

    # Training completed successfully
    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"✅ BASELINE GRPO TRAINING COMPLETED")
    print(f"{'='*60}")
    print(f"   Training time: {elapsed/3600:.2f} hours")
    print(f"   Final checkpoint: {trainer_baseline.state.global_step} steps")
    print(f"   Final loss: {trainer_baseline.state.log_history[-1]['loss']:.4f}")
    print(f"{'='*60}")

except KeyboardInterrupt:
    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"⚠️  TRAINING INTERRUPTED BY USER")
    print(f"{'='*60}")
    print(f"   Training time so far: {elapsed/3600:.2f} hours")
    print(f"   Last checkpoint: {trainer_baseline.state.global_step}")
    print(f"   Can resume from checkpoint later")
    print(f"{'='*60}")

except Exception as e:
    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"❌ TRAINING FAILED")
    print(f"{'='*60}")
    print(f"   Error: {e}")
    print(f"   Training time before failure: {elapsed/3600:.2f} hours")
    print(f"   Last checkpoint: {trainer_baseline.state.global_step}")
    print(f"{'='*60}")
    raise

# Save training metadata
metadata_baseline = {
    "model": "Qwen2.5-Math-1.5B",
    "training_type": "Baseline GRPO",
    "dataset": "NuminaMath-CoT" if "NuminaMath" in str(type(train_dataset_full)) else "GSM8K",
    "num_prompts": len(train_prompts_full),
    "final_step": trainer_baseline.state.global_step,
    "training_time_hours": elapsed / 3600,
    "final_loss": float(trainer_baseline.state.log_history[-1]['loss']),
    "hyperparameters": {
        "learning_rate": grpo_config.learning_rate,
        "num_generations": grpo_config.num_generations,
        "batch_size": grpo_config.per_device_train_batch_size,
        "gradient_accumulation_steps": grpo_config.gradient_accumulation_steps,
    }
}

with open("./baseline_grpo_checkpoints/training_metadata.json", "w") as f:
    json.dump(metadata_baseline, f, indent=2)

print("\n✅ Training metadata saved to training_metadata.json")


🚀 BASELINE GRPO TRAINING STARTED
   Time: 2026-05-15 04:01:28.345492
   Dataset: 1024 prompts
   Generations per prompt: 8
   Expected duration: ~30-40 GPU hours
   Checkpoints: every 200 steps → ./baseline_grpo_checkpoints/

⚠️  MONITORING CHECKLIST (check every ~2 hours):
   ✅ Training loss decreasing
   ✅ VRAM stable (not creeping up)
   ✅ Reward distribution: mix of +1.0 and -1.0
   ✅ Checkpoints saving every 200 steps
   ⚠️  If loss is NaN → stop and debug
   ⚠️  If all rewards are -1 → verifier broken


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,024 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
Both `max_new_tokens` (=4096) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/py


❌ TRAINING FAILED
   Error: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 981.81 MiB is free. Including non-PyTorch memory, this process has 13.60 GiB memory in use. Of the allocated memory 13.15 GiB is allocated by PyTorch, and 301.53 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
   Training time before failure: 0.25 hours
   Last checkpoint: 0


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 981.81 MiB is free. Including non-PyTorch memory, this process has 13.60 GiB memory in use. Of the allocated memory 13.15 GiB is allocated by PyTorch, and 301.53 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
print(f"\n{'='*60}")
print(f"🎉 DAY 2 COMPLETE")
print(f"{'='*60}")
print(f"✅ Baseline GRPO training finished")
print(f"✅ Model checkpoints saved")
print(f"✅ Training metadata logged")
print(f"\n🚀 NEXT: Day 3 - Launch PROF-GRPO training in parallel")
print(f"{'='*60}") 